# Решения: практикум агрегатов

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-5

In [ ]:
orders_pay = orders.merge(payments, on='order_id', how='left')
orders_full = orders_pay.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
ref_date = orders['order_purchase_timestamp'].max()
rfm = (
    orders_full.groupby('customer_id')
    .agg(
        last_purchase=('order_purchase_timestamp', 'max'),
        Frequency=('order_id', 'nunique'),
        Monetary=('payment_value', 'sum'),
    )
    .reset_index()
)
rfm['Recency'] = (ref_date - rfm['last_purchase']).dt.days
card_share = orders_full.groupby('customer_id')['payment_type'].apply(lambda s: float((s == 'credit_card').mean()))
delivery = orders_full.assign(
    days_to_deliver=(orders_full['order_delivered_customer_date'] - orders_full['order_purchase_timestamp']).dt.days
).groupby('customer_id')['days_to_deliver'].mean()
rfm_plus = rfm.merge(card_share.rename('share_card'), on='customer_id', how='left')
rfm_plus = rfm_plus.merge(delivery.rename('avg_days_to_deliver'), on='customer_id', how='left')
rfm_plus = rfm_plus[['customer_id', 'Recency', 'Frequency', 'Monetary', 'share_card', 'avg_days_to_deliver']]
n_notna_delay = int(rfm_plus['avg_days_to_deliver'].notna().sum())
corr_fm = float(rfm_plus['Frequency'].corr(rfm_plus['Monetary']))
CORR_NOTE = (
    'Связь F и M обычно положительная: больше заказов — выше суммарная выручка. '
    'Но корреляция не идеальна, потому что у части клиентов редкие, но дорогие заказы.'
)
print(rfm_plus.head())
print('corr:', round(corr_fm, 3), 'rows with delivery avg:', n_notna_delay)
print(CORR_NOTE)

## ДЗ. 1-3

In [ ]:
orders_pay = orders.merge(payments, on='order_id', how='left')
orders_full = orders_pay.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
ref_date = orders['order_purchase_timestamp'].max()
rfm = (
    orders_full.groupby('customer_id')
    .agg(last_purchase=('order_purchase_timestamp', 'max'), Frequency=('order_id', 'nunique'), Monetary=('payment_value', 'sum'))
    .reset_index()
)
rfm['Recency'] = (ref_date - rfm['last_purchase']).dt.days
customer_states = orders_full.groupby('customer_id')['customer_state'].nunique()
n_states = customer_states
scored = rfm.copy()
scored['score'] = scored['Monetary'] / scored['Monetary'].mean() + scored['Frequency'] / scored['Frequency'].mean()
top10 = scored.sort_values('score', ascending=False).head(10)
SCALE_NOTE = (
    'Recency измеряется в днях, Monetary — в денежных единицах, Frequency — в штуках. '
    'Если складывать их как есть, признак с большим масштабом доминирует. '
    'Перед единым score нужно нормировать или явно задавать веса.'
)
print(n_states.describe())
print(top10[['customer_id', 'score']])
print(SCALE_NOTE)